In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\arjav\DevSource\toc-performance-dashboard")

DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
TOC_DIR = BOUNDARIES_DIR / "tocs"
VILLAGE_DIR = BOUNDARIES_DIR / "villages"
JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
PARCELS_DIR = DATA_DIR / "parcels" / "processed"

toc_clean_path = BOUNDARIES_DIR / "processed" / "toc_clean.gpkg"

toc_demographics_path = TOC_DIR / "toc_demographics.gpkg"

toc_streets_path = TOC_DIR / "toc_full_street_miles.gpkg"

toc_jobs_path = JOBS_DIR / "toc_jobs_2022.gpkg"

toc_parcels_path = PARCELS_DIR / "parcels_assessor_toc.gpkg"

toc_clean_path, toc_demographics_path, toc_streets_path, toc_jobs_path, toc_parcels_path

(WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/toc_demographics.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/toc_full_street_miles.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/toc_jobs_2022.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/parcels_assessor_toc.gpkg'))

In [2]:
toc_clean = gpd.read_file(toc_clean_path)

In [3]:
toc_demographics = gpd.read_file(toc_demographics_path)

In [4]:
toc_streets = gpd.read_file(toc_streets_path)

In [5]:
toc_jobs = gpd.read_file(toc_jobs_path)

In [6]:
toc_parcels = gpd.read_file(toc_parcels_path)

In [7]:
print(toc_clean.columns)
print(toc_demographics.columns)
print(toc_streets.columns)
print(toc_jobs.columns)
print(toc_parcels.columns)

Index(['OBJECTID', 'APPLICABIL', 'geometry'], dtype='object')
Index(['OBJECTID', 'APPLICABIL', 'TOC_ID', 'income_weighted', 'rent_weighted',
       'hh_total', 'median_income_approx', 'median_rent_approx',
       'intersect_acres', 'ASNQE001', 'ASS8E001', 'ASS8E002', 'ASS8E003',
       'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003', 'ASORE004', 'ASORE010',
       'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002', 'pct_drive_alone',
       'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto',
       'geometry'],
      dtype='object')
Index(['OBJECTID_x', 'APPLICABIL', 'TOC_ID', 'income_weighted',
       'rent_weighted', 'hh_total', 'median_income_approx',
       'median_rent_approx', 'intersect_acres', 'ASNQE001', 'ASS8E001',
       'ASS8E002', 'ASS8E003', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002',
       'pct_drive_alone', 'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh',
       'pct_auto', 'OBJECT

In [8]:
# lots of data cleaning needed here, quite a few duplicates... and I think this parcel data isn't actually aggregated the way it should be. let's do that.

toc_base = toc_streets.copy()

# Drop duplicate object IDs
drop_cols = [c for c in ["OBJECTID_x", "OBJECTID_y"] if c in toc_base.columns]
toc_base = toc_base.drop(columns=drop_cols)

toc_base.columns

Index(['APPLICABIL', 'TOC_ID', 'income_weighted', 'rent_weighted', 'hh_total',
       'median_income_approx', 'median_rent_approx', 'intersect_acres',
       'ASNQE001', 'ASS8E001', 'ASS8E002', 'ASS8E003', 'ASS9E002', 'ASS9E003',
       'ASORE001', 'ASORE003', 'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019',
       'ASORE021', 'ASORE002', 'pct_drive_alone', 'pct_transit', 'pct_bike',
       'pct_walk', 'pct_wfh', 'pct_auto', 'street_miles_total',
       'street_miles_per_acre', 'geometry'],
      dtype='object')

In [9]:
# aggregate parcel values by TOC applicability

value_agg = (
    toc_parcels
    .groupby("APPLICABIL", as_index=False)
    .agg({
        "FULLCASHVALUE": "sum",
        "LANDFULLCASHVALUE": "sum",
        "IMPRFULLCASHVALUE": "sum",
        "LPVAMOUNT": "sum",
        "LPVASSESSEDVALUE": "sum",
        "SQFOOTAGE": "sum",
        "NUMBEROFUNITS": "sum",
    })
)

value_agg.head()

,APPLICABIL,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS
0,50th Street Station Area,1.238704e+09,2.995826e+08,9.391213e+08,5.086285e+08,74683387.0,32849304.0,35.0
1,Capitol Extension Area,1.961782e+09,5.294767e+08,1.432305e+09,1.030840e+09,148353832.0,34651611.0,602.0
2,I-10 West Extension Area,8.804660e+09,1.985700e+09,6.818960e+09,3.439952e+09,468557108.0,266256494.0,474.0
3,Northwest Extension Area II,2.508616e+09,7.254658e+08,1.783150e+09,1.027264e+09,149286869.0,61353747.0,119.0
4,TOD District - 19North,3.933583e+09,8.894743e+08,3.044109e+09,1.448683e+09,168607085.0,70133705.0,98.0


In [10]:
jobs_trim = toc_jobs.drop(columns=["geometry", "OBJECTID"], errors="ignore")

jobs_trim.head()

,TOC_ID,APPLICABIL,jobs_total,acres_contributing,toc_area_sqft,toc_acres,jobs_per_acre
0,0,TOD District - Gateway,4984.867123,2418.944362,1.053692e+08,2418.944362,2.060761
1,1,TOD District - Eastlake Garfield,6847.046904,1220.462979,5.316337e+07,1220.462979,5.610205
2,2,TOD District - Midtown,7099.922548,1283.000217,5.588749e+07,1283.000217,5.533844
3,3,TOD District - Uptown,441.878230,1332.064285,5.802472e+07,1332.064285,0.331724
4,4,TOD District - Solano,2006.609086,1100.265433,4.792756e+07,1100.265433,1.823750


In [11]:
toc_final = (
    toc_base
    # add jobs metrics
    .merge(
        jobs_trim,
        on=["TOC_ID", "APPLICABIL"],
        how="left",
        suffixes=("", "_jobs")
    )
    # add assessor / value metrics
    .merge(
        value_agg,
        on="APPLICABIL",
        how="left",
        suffixes=("", "_assessor")
    )
)

toc_final.head()

,APPLICABIL,TOC_ID,income_weighted,rent_weighted,hh_total,median_income_approx,median_rent_approx,intersect_acres,ASNQE001,ASS8E001,...,toc_area_sqft,toc_acres,jobs_per_acre,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS
0,TOD District - Gateway,0,2.099283e+08,5.097215e+06,4404.074570,47666.824758,1157.386086,2418.944362,13919.344291,4802.854853,...,1.053692e+08,2418.944362,2.060761,3.718919e+09,8.830583e+08,2.835861e+09,1.930846e+09,285984758.0,102963911.0,727.0
1,TOD District - Eastlake Garfield,1,1.613483e+08,3.709420e+06,3379.818960,47738.728035,1097.520452,1220.462979,7826.045040,4021.409613,...,5.316337e+07,1220.462979,5.610205,3.014334e+09,8.307006e+08,2.183633e+09,1.502193e+09,207168885.0,44165771.0,509.0
2,TOD District - Midtown,2,5.228343e+08,1.150085e+07,7079.314543,73853.809662,1624.571152,1283.000217,11830.041147,8259.602020,...,5.588749e+07,1283.000217,5.533844,7.310771e+09,1.939664e+09,5.371106e+09,3.502604e+09,468187974.0,45643087.0,424.0
3,TOD District - Uptown,3,4.255024e+08,8.094779e+06,5546.764459,76711.815672,1459.369509,1332.064285,10889.582934,5910.686353,...,5.802472e+07,1332.064285,0.331724,3.589014e+09,9.414611e+08,2.647553e+09,1.471534e+09,171863623.0,45244923.0,270.0
4,TOD District - Solano,4,2.998474e+08,7.064961e+06,5518.487681,54335.073430,1280.234957,1100.265433,14440.725434,6067.435580,...,4.792756e+07,1100.265433,1.823750,1.978354e+09,4.115623e+08,1.566791e+09,7.172634e+08,91072483.0,38193687.0,120.0


In [12]:
# Guard against zero/NaN area
valid_area = toc_final["toc_acres"].replace(0, pd.NA)

toc_final["taxable_value_per_acre"] = toc_final["LPVASSESSEDVALUE"] / valid_area
toc_final["full_value_per_acre"]    = toc_final["FULLCASHVALUE"]    / valid_area

# Optional: land-only value per acre
toc_final["land_value_per_acre"]    = toc_final["LANDFULLCASHVALUE"] / valid_area


In [13]:
toc_final[[
    "APPLICABIL",
    "toc_acres",
    "jobs_total",
    "jobs_per_acre",
    "LPVASSESSEDVALUE",
    "taxable_value_per_acre"
]].head(15)

,APPLICABIL,toc_acres,jobs_total,jobs_per_acre,LPVASSESSEDVALUE,taxable_value_per_acre
0,TOD District - Gateway,2418.944362,4984.867123,2.060761,285984758.0,118227.092166
1,TOD District - Eastlake Garfield,1220.462979,6847.046904,5.610205,207168885.0,169746.144404
2,TOD District - Midtown,1283.000217,7099.922548,5.533844,468187974.0,364916.519800
3,TOD District - Uptown,1332.064285,441.878230,0.331724,171863623.0,129020.517213
4,TOD District - Solano,1100.265433,2006.609086,1.823750,91072483.0,82773.192961
5,TOD District - 19North,1780.947386,2635.120908,1.479617,168607085.0,94672.692915
6,50th Street Station Area,617.105659,1628.549532,2.639012,74683387.0,121022.042054
7,Capitol Extension Area,1116.466203,3529.734836,3.161524,148353832.0,132878.032155
8,I-10 West Extension Area,7735.922063,6465.078133,0.835722,468557108.0,60569.005763
9,Northwest Extension Area II,1740.704483,1633.684459,0.938519,149286869.0,85762.328100


In [14]:
# finally getting around to renaming columns since people will have to look at this in some capacity... oops

toc_final = toc_final.rename(columns={
    "APPLICABIL": "TOC_Name",
    "ASNQE001": "Total_Pop",
    "ASS8E002": "Occupied_Units",
    "ASS8E003": "Vacant_Units",
    "median_income_approx": "Median_Income",
    "median_rent_approx": "Median_Rent",
    "street_miles_total": "Street_Miles_Total",
    "street_miles_per_acre": "Street_Miles_per_Acre",
    "jobs_total": "Jobs_Total",
    "jobs_per_acre": "Jobs_per_Acre",
    "ASNQE001":   "Total_Pop",
    "ASS8E001":   "Total_Housing_Units",
    "ASS8E002":   "Occupied_Units",
    "ASS8E003":   "Vacant_Units",
    "ASS9E002":   "Owner_Occupied",
    "ASS9E003":   "Renter_Occupied",
    "ASORE001":   "Workers_16_plus",
    "ASORE003":   "Workers_Drive_Alone",
    "ASORE004":   "Workers_Carpool",
    "ASORE010":   "Workers_Transit",
    "ASORE018":   "Workers_Bike",
    "ASORE019":   "Workers_Walk",
    "ASORE021":   "Workers_WFH",
    "ASORE002":   "Workers_Auto_Total"
})

In [15]:
toc_final.head()

,TOC_Name,TOC_ID,income_weighted,rent_weighted,hh_total,Median_Income,Median_Rent,intersect_acres,Total_Pop,Total_Housing_Units,...,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,taxable_value_per_acre,full_value_per_acre,land_value_per_acre
0,TOD District - Gateway,0,2.099283e+08,5.097215e+06,4404.074570,47666.824758,1157.386086,2418.944362,13919.344291,4802.854853,...,3.718919e+09,8.830583e+08,2.835861e+09,1.930846e+09,285984758.0,102963911.0,727.0,118227.092166,1.537414e+06,3.650593e+05
1,TOD District - Eastlake Garfield,1,1.613483e+08,3.709420e+06,3379.818960,47738.728035,1097.520452,1220.462979,7826.045040,4021.409613,...,3.014334e+09,8.307006e+08,2.183633e+09,1.502193e+09,207168885.0,44165771.0,509.0,169746.144404,2.469828e+06,6.806438e+05
2,TOD District - Midtown,2,5.228343e+08,1.150085e+07,7079.314543,73853.809662,1624.571152,1283.000217,11830.041147,8259.602020,...,7.310771e+09,1.939664e+09,5.371106e+09,3.502604e+09,468187974.0,45643087.0,424.0,364916.519800,5.698184e+06,1.511819e+06
3,TOD District - Uptown,3,4.255024e+08,8.094779e+06,5546.764459,76711.815672,1459.369509,1332.064285,10889.582934,5910.686353,...,3.589014e+09,9.414611e+08,2.647553e+09,1.471534e+09,171863623.0,45244923.0,270.0,129020.517213,2.694325e+06,7.067685e+05
4,TOD District - Solano,4,2.998474e+08,7.064961e+06,5518.487681,54335.073430,1280.234957,1100.265433,14440.725434,6067.435580,...,1.978354e+09,4.115623e+08,1.566791e+09,7.172634e+08,91072483.0,38193687.0,120.0,82773.192961,1.798070e+06,3.740573e+05


In [16]:
toc_final.columns

Index(['TOC_Name', 'TOC_ID', 'income_weighted', 'rent_weighted', 'hh_total',
       'Median_Income', 'Median_Rent', 'intersect_acres', 'Total_Pop',
       'Total_Housing_Units', 'Occupied_Units', 'Vacant_Units',
       'Owner_Occupied', 'Renter_Occupied', 'Workers_16_plus',
       'Workers_Drive_Alone', 'Workers_Carpool', 'Workers_Transit',
       'Workers_Bike', 'Workers_Walk', 'Workers_WFH', 'Workers_Auto_Total',
       'pct_drive_alone', 'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh',
       'pct_auto', 'Street_Miles_Total', 'Street_Miles_per_Acre', 'geometry',
       'Jobs_Total', 'acres_contributing', 'toc_area_sqft', 'toc_acres',
       'Jobs_per_Acre', 'FULLCASHVALUE', 'LANDFULLCASHVALUE',
       'IMPRFULLCASHVALUE', 'LPVAMOUNT', 'LPVASSESSEDVALUE', 'SQFOOTAGE',
       'NUMBEROFUNITS', 'taxable_value_per_acre', 'full_value_per_acre',
       'land_value_per_acre'],
      dtype='object')

In [17]:
# and time to save the final output..

output_path = OUTPUT_DIR / "toc_dashboard.gpkg"

toc_final.to_file(output_path, driver="GPKG")

print("Saved TOC dashboard layer:", output_path)

Saved TOC dashboard layer: C:\Users\arjav\DevSource\toc-performance-dashboard\outputs\toc_dashboard.gpkg


In [23]:
# Normally, at this point, I would use arcpy to publish the layer to AGOL as a hosted feature layer. However, because these were all saved as GeoPackages, and publishing those would require converting all of them to SEDFs first, I am instead opting to publish directly using the AGOL web interface.

# Which means that we have officially concluded the compilation process, and as a result, the Python portion of this project. But first:

print(f"*mic drop* Rawal out.")

*mic drop* Rawal out.
